In [1]:
import pandas as pd

df = pd.read_csv("../datos/crudos/MEX.csv")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)  # viene dd/mm/yyyy
print(df.shape)
df.head()

(4743, 25)


,Country,League,Season,Date,Time,Home,Away,HG,AG,Res,...,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA
0,Mexico,Liga MX,2012/2013,2012-07-21,01:30,Chiapas,Tigres UANL,0,4,A,...,2.64,2.74,3.10,2.51,NaN,NaN,NaN,NaN,NaN,NaN
1,Mexico,Liga MX,2012/2013,2012-07-21,03:30,Club Tijuana,Puebla,2,0,H,...,4.49,1.93,3.26,3.83,NaN,NaN,NaN,NaN,NaN,NaN
2,Mexico,Liga MX,2012/2013,2012-07-21,23:00,Cruz Azul,Monarcas,0,0,D,...,4.13,2.00,3.26,3.56,NaN,NaN,NaN,NaN,NaN,NaN
3,Mexico,Liga MX,2012/2013,2012-07-21,23:00,Queretaro,Club Leon,0,2,A,...,2.29,3.26,3.18,2.16,NaN,NaN,NaN,NaN,NaN,NaN
4,Mexico,Liga MX,2012/2013,2012-07-22,01:00,Monterrey,Club America,0,0,D,...,4.50,1.92,3.37,3.79,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
print(df["Season"].value_counts().sort_index())
print(df["Res"].value_counts(normalize=True).round(3))
print(df.isna().sum())

Season
2012/2013    334
2013/2014    334
2014/2015    334
2015/2016    334
2016/2017    334
2017/2018    334
2018/2019    334
2019/2020    275
2020/2021    342
2021/2022    342
2022/2023    342
2023/2024    340
2024/2025    340
2025/2026    336
2026/2027     88
Name: count, dtype: int64
Res
H    0.446
A    0.283
D    0.271
Name: proportion, dtype: float64
Country       0
League        0
Season        0
Date          0
Time          0
Home          0
Away          0
HG            0
AG            0
Res           0
PSCH        306
PSCD        306
PSCA        306
MaxCH         0
MaxCD         0
MaxCA         0
AvgCH         0
AvgCD         0
AvgCA         0
BFECH      4020
BFECD      4020
BFECA      4020
B365CH     4354
B365CD     4354
B365CA     4354
dtype: int64


In [3]:
print(df["Home"].nunique(), "equipos")
print(sorted(df["Home"].unique()))
print(df["Date"].min(), "->", df["Date"].max())

25 equipos
['Atl. San Luis', 'Atlante', 'Atlas', 'Chiapas', 'Club America', 'Club Leon', 'Club Tijuana', 'Cruz Azul', 'Dorados de Sinaloa', 'Guadalajara Chivas', 'Juarez', 'Leones Negros', 'Lobos BUAP', 'Mazatlan FC', 'Monarcas', 'Monterrey', 'Necaxa', 'Pachuca', 'Puebla', 'Queretaro', 'Santos Laguna', 'Tigres UANL', 'Toluca', 'UNAM Pumas', 'Veracruz']
2012-07-21 00:00:00 -> 2026-09-28 00:00:00


In [4]:
print(df[["PSCH", "PSCD", "PSCA", "AvgCH", "AvgCD", "AvgCA"]].isna().sum().to_string())
print(df.groupby("Season")["PSCH"].apply(lambda s: s.notna().sum()).to_string())
print(df["Home"].value_counts().sort_index().to_string())

PSCH     306
PSCD     306
PSCA     306
AvgCH      0
AvgCD      0
AvgCA      0
Season
2012/2013    334
2013/2014    334
2014/2015    334
2015/2016    333
2016/2017    331
2017/2018    333
2018/2019    332
2019/2020    270
2020/2021    342
2021/2022    341
2022/2023    342
2023/2024    336
2024/2025    340
2025/2026    135
2026/2027      0
Home
Atl. San Luis         145
Atlante                39
Atlas                 255
Chiapas                87
Club America          294
Club Leon             269
Club Tijuana          257
Cruz Azul             275
Dorados de Sinaloa     17
Guadalajara Chivas    263
Juarez                123
Leones Negros          17
Lobos BUAP             34
Mazatlan FC           102
Monarcas              143
Monterrey             279
Necaxa                180
Pachuca               268
Puebla                250
Queretaro             244
Santos Laguna         266
Tigres UANL           287
Toluca                274
UNAM Pumas            262
Veracruz              113


In [5]:
# columnas útiles y orden cronológico (clave para no mezclar futuro con pasado)
cols = ["Season", "Date", "Home", "Away", "HG", "AG", "Res", "AvgCH", "AvgCD", "AvgCA"]
d = df[cols].copy().sort_values("Date").reset_index(drop=True)

# momios -> probabilidades implícitas, quitando el margen de la casa
inv = 1 / d[["AvgCH", "AvgCD", "AvgCA"]]
prob = inv.div(inv.sum(axis=1), axis=0)
prob.columns = ["pH", "pD", "pA"]
d = pd.concat([d, prob], axis=1)

# predicción del mercado = el resultado con mayor probabilidad
d["mercado"] = prob.idxmax(axis=1).map({"pH": "H", "pD": "D", "pA": "A"})

print("Siempre local:", (d["Res"] == "H").mean().round(3))
print("Mercado:", (d["mercado"] == d["Res"]).mean().round(3))
print(d.assign(ok=d["mercado"] == d["Res"]).groupby("Season")["ok"].mean().round(3).to_string())
print(d["mercado"].value_counts().to_string())
print(set(df["Home"]) ^ set(df["Away"]))  # debe salir set() vacío

Siempre local: 0.446
Mercado: 0.488
Season
2012/2013    0.485
2013/2014    0.476
2014/2015    0.431
2015/2016    0.473
2016/2017    0.449
2017/2018    0.452
2018/2019    0.542
2019/2020    0.462
2020/2021    0.529
2021/2022    0.439
2022/2023    0.506
2023/2024    0.512
2024/2025    0.541
2025/2026    0.533
2026/2027    0.455
mercado
H    3527
A    1211
D       5
set()


In [6]:
import numpy as np
from collections import defaultdict, deque

K, VENTAJA_LOCAL = 20, 60          # parámetros de Elo, se pueden afinar después
elo = defaultdict(lambda: 1500.0)
hist = defaultdict(lambda: deque(maxlen=5))   # (puntos, goles a favor, goles en contra)

def resumen(q):
    return np.array(q).mean(axis=0) if len(q) else np.array([np.nan] * 3)

filas = []
for r in d.itertuples():
    h, a = r.Home, r.Away
    ph, gfh, gch = resumen(hist[h])
    pa, gfa, gca = resumen(hist[a])
    filas.append(dict(elo_h=elo[h], elo_a=elo[a],
                      pts_h=ph, gf_h=gfh, gc_h=gch, n_h=len(hist[h]),
                      pts_a=pa, gf_a=gfa, gc_a=gca, n_a=len(hist[a])))

    # actualizar DESPUÉS de guardar las features del partido
    s_h = {"H": 1.0, "D": 0.5, "A": 0.0}[r.Res]
    esperado = 1 / (1 + 10 ** ((elo[a] - (elo[h] + VENTAJA_LOCAL)) / 400))
    delta = K * (s_h - esperado)
    elo[h] += delta
    elo[a] -= delta
    pts_h = {"H": 3, "D": 1, "A": 0}[r.Res]
    pts_a = {"H": 0, "D": 1, "A": 3}[r.Res]
    hist[h].append((pts_h, r.HG, r.AG))
    hist[a].append((pts_a, r.AG, r.HG))

feat = pd.concat([d, pd.DataFrame(filas)], axis=1)
feat["elo_diff"] = feat["elo_h"] - feat["elo_a"]
feat["pts_diff"] = feat["pts_h"] - feat["pts_a"]

feat.to_csv("../datos/procesados/partidos.csv", index=False)
print(feat.shape)
print(feat[["Date", "Home", "Away", "elo_h", "elo_a", "pts_h", "n_h"]].head(3).to_string())
print(feat[["Date", "Home", "Away", "elo_h", "elo_a", "pts_h", "n_h"]].tail(3).to_string())

(4743, 26)
        Date          Home         Away   elo_h   elo_a  pts_h  n_h
0 2012-07-21       Chiapas  Tigres UANL  1500.0  1500.0    NaN    0
1 2012-07-21  Club Tijuana       Puebla  1500.0  1500.0    NaN    0
2 2012-07-21     Cruz Azul     Monarcas  1500.0  1500.0    NaN    0
           Date        Home           Away        elo_h        elo_a  pts_h  n_h
4740 2026-09-27  UNAM Pumas  Atl. San Luis  1584.097854  1459.899852    1.0    5
4741 2026-09-28   Club Leon         Juarez  1495.645821  1426.658042    1.6    5
4742 2026-09-28      Necaxa   Club America  1455.271176  1636.269088    0.4    5


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier

feat = pd.read_csv("../datos/procesados/partidos.csv", parse_dates=["Date"])
feat["y"] = feat["Res"].map({"A": 0, "D": 1, "H": 2})

X_cols = ["elo_diff", "elo_h", "elo_a", "pts_h", "pts_a", "gf_h", "gc_h", "gf_a", "gc_a"]

# 2012/13 se usa solo para "calentar" el Elo; el 2026/27 en curso queda fuera como prueba en vivo
train = feat[feat["Season"].between("2013/2014", "2022/2023")]
test = feat[feat["Season"].between("2023/2024", "2025/2026")]
print(len(train), "train |", len(test), "test")

modelos = {
    "logistica": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                               LogisticRegression(max_iter=1000)),
    "xgboost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             eval_metric="mlogloss", random_state=42),
}

y_test = test["y"].values
res = {}

# baseline 1: frecuencias históricas del train
prior = train["y"].value_counts(normalize=True).sort_index().values
p = np.tile(prior, (len(test), 1))
res["frecuencias"] = (log_loss(y_test, p, labels=[0, 1, 2]), accuracy_score(y_test, p.argmax(axis=1)))

# baseline 2: el mercado (columnas en orden A, D, H = 0, 1, 2)
p = test[["pA", "pD", "pH"]].values
res["mercado"] = (log_loss(y_test, p, labels=[0, 1, 2]), accuracy_score(y_test, p.argmax(axis=1)))

for nombre, m in modelos.items():
    m.fit(train[X_cols], train["y"])
    p = m.predict_proba(test[X_cols])
    res[nombre] = (log_loss(y_test, p, labels=[0, 1, 2]), accuracy_score(y_test, p.argmax(axis=1)))

print(pd.DataFrame(res, index=["log_loss", "accuracy"]).T.round(4).sort_values("log_loss").to_string())

3305 train | 1016 test
             log_loss  accuracy
mercado        0.9809    0.5285
logistica      1.0090    0.5049
xgboost        1.0213    0.5020
frecuencias    1.0599    0.4705


In [8]:
X_mkt = X_cols + ["pH", "pD", "pA"]
m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                  LogisticRegression(max_iter=1000))
m.fit(train[X_mkt], train["y"])
p = m.predict_proba(test[X_mkt])
print("logistica + mercado:",
      round(log_loss(y_test, p, labels=[0, 1, 2]), 4),
      round(accuracy_score(y_test, p.argmax(axis=1)), 4))

logistica + mercado: 0.9879 0.5197


In [9]:
from itertools import product

MAP = {"A": 0, "D": 1, "H": 2}

def procesar(d, K=20, ventaja=60, regresion=0.0):
    elo = defaultdict(lambda: 1500.0)
    hist = defaultdict(lambda: deque(maxlen=5))
    filas, temp = [], None
    for r in d.itertuples():
        if r.Season != temp:                      # nueva temporada: regresión a la media
            if temp is not None:
                for t in list(elo):
                    elo[t] = 1500 + (1 - regresion) * (elo[t] - 1500)
            temp = r.Season
        h, a = r.Home, r.Away
        ph, gfh, gch = resumen(hist[h])
        pa, gfa, gca = resumen(hist[a])
        filas.append(dict(elo_h=elo[h], elo_a=elo[a], pts_h=ph, gf_h=gfh, gc_h=gch,
                          pts_a=pa, gf_a=gfa, gc_a=gca))
        s = {"H": 1.0, "D": 0.5, "A": 0.0}[r.Res]
        esp = 1 / (1 + 10 ** ((elo[a] - (elo[h] + ventaja)) / 400))
        elo[h] += K * (s - esp)
        elo[a] -= K * (s - esp)
        hist[h].append(({"H": 3, "D": 1, "A": 0}[r.Res], r.HG, r.AG))
        hist[a].append(({"H": 0, "D": 1, "A": 3}[r.Res], r.AG, r.HG))
    out = pd.concat([d.reset_index(drop=True), pd.DataFrame(filas)], axis=1)
    out["elo_diff"] = out["elo_h"] - out["elo_a"]
    out["y"] = out["Res"].map(MAP)
    return out, elo, hist

def log_loss_val(K, v, rg):
    f, _, _ = procesar(d, K, v, rg)
    tr = f[f["Season"].between("2013/2014", "2020/2021")]
    va = f[f["Season"].between("2021/2022", "2022/2023")]
    m = LogisticRegression(max_iter=1000).fit(tr[["elo_diff"]], tr["y"])
    return log_loss(va["y"], m.predict_proba(va[["elo_diff"]]), labels=[0, 1, 2])

rej = [(K, v, rg, log_loss_val(K, v, rg))
       for K, v, rg in product([10, 20, 30, 40], [0, 40, 60, 100], [0, 0.25, 0.5])]
print(pd.DataFrame(rej, columns=["K", "ventaja", "regresion", "log_loss"])
        .sort_values("log_loss").head(5).to_string(index=False))

 K  ventaja  regresion  log_loss
10      100       0.25  1.055530
10       60       0.25  1.055716
10       40       0.25  1.055814
10      100       0.00  1.055871
10       60       0.00  1.055939


In [10]:
K, v, rg = 20, 60, 0.0    # <- cambia por los mejores de la rejilla

feat2, elo_fin, hist_fin = procesar(d, K, v, rg)
tr = feat2[feat2["Season"].between("2013/2014", "2022/2023")]
te = feat2[feat2["Season"].between("2023/2024", "2025/2026")]
m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
m.fit(tr[X_cols], tr["y"])
print("antes: 1.0090 | ahora:", round(log_loss(te["y"], m.predict_proba(te[X_cols]), labels=[0, 1, 2]), 4))

antes: 1.0090 | ahora: 1.009


In [11]:
import joblib

final = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
final.fit(feat2[X_cols], feat2["y"])

ultima = feat2["Season"].max()
activos = sorted(set(feat2.loc[feat2["Season"] == ultima, "Home"]) | set(feat2.loc[feat2["Season"] == ultima, "Away"]))

joblib.dump({"modelo": final, "cols": X_cols,
             "elo": dict(elo_fin),
             "hist": {t: list(q) for t, q in hist_fin.items()},   # defaultdict con lambda no se puede guardar
             "activos": activos},
            "../modelos/modelo.joblib")
print(len(activos), "equipos activos:", activos)

18 equipos activos: ['Atl. San Luis', 'Atlante', 'Atlas', 'Club America', 'Club Leon', 'Club Tijuana', 'Cruz Azul', 'Guadalajara Chivas', 'Juarez', 'Monterrey', 'Necaxa', 'Pachuca', 'Puebla', 'Queretaro', 'Santos Laguna', 'Tigres UANL', 'Toluca', 'UNAM Pumas']


In [13]:
ls modelos


 El volumen de la unidad C no tiene etiqueta.
 El n�mero de serie del volumen es: 1280-642B

 Directorio de c:\Users\Usuario\ligamx-match-predictor\cuadernos



No se encuentra el archivo
